# init

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers torch pillow openai


In [ ]:
import pandas as pd
import numpy as np
import json
import faiss
import torch
from PIL import Image
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# ========================== CONFIGURATION ==========================
@dataclass
class RAGConfig:
    csv_path: str = '/content/RangoliDataset_6cat.csv'
    clip_model_name: str = 'openai/clip-vit-base-patch32'
    top_k: int = 3
    confidence_threshold: float = 0.22  # CLIP image-to-text threshold
    llm_temperature: float = 0.1
    llm_max_tokens: int = 1000
    llm_provider: str = "none"  # Options: 'none', 'gemini'
    gemini_api_key: str = ""    # For Google Gemini (aistudio.google.com)
    gemini_model: str = "gemini-2.0-flash"  # Current model (Dec 2025)
    debug: bool = True

CONFIG = RAGConfig()

# ========================== MODEL LOADING ==========================
CLIP_MODEL, CLIP_PROCESSOR, DEVICE = None, None, None

def get_clip_model():
    global CLIP_MODEL, CLIP_PROCESSOR, DEVICE
    if CLIP_MODEL is None:
        from transformers import CLIPProcessor, CLIPModel
        print("🔄 Loading CLIP model...")
        CLIP_MODEL = CLIPModel.from_pretrained(CONFIG.clip_model_name)
        CLIP_PROCESSOR = CLIPProcessor.from_pretrained(CONFIG.clip_model_name)
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        CLIP_MODEL = CLIP_MODEL.to(DEVICE)
        CLIP_MODEL.eval()
        print(f"✅ CLIP model loaded on {DEVICE}")
    return CLIP_MODEL, CLIP_PROCESSOR, DEVICE

# ========================== DATASET ==========================
def load_category_dataset(csv_path: str) -> pd.DataFrame:
    print(f"📂 Loading category dataset from {csv_path}...")
    df = pd.read_csv(csv_path)
    df = df.fillna('')

    # Create CLIP-optimized visual description for each category
    def create_clip_prompt(row) -> str:
        """Create a visual description that CLIP can match against images."""
        category = row.get('Category', '')
        local_name = row.get('Local_Name', '')
        visual_desc = row.get('Visual_Description', '')
        unique_ids = row.get('Unique_Identifiers', '')
        colors = row.get('Traditional_Colours', '')
        pattern = row.get('Design_Pattern', '')

        # Build concise visual prompt for CLIP (max 77 tokens)
        prompt = f"A traditional Indian floor art called {local_name}. {visual_desc[:200]}"
        return prompt

    df['clip_prompt'] = df.apply(create_clip_prompt, axis=1)
    df['category_id'] = df.index.tolist()

    print(f"✅ Loaded {len(df)} categories:")
    for _, row in df.iterrows():
        print(f"   • {row['Category']} ({row['State']}) - {row['Local_Name']}")

    return df

# ========================== EMBEDDING ==========================
def embed_texts(texts: List[str], model, processor, device) -> np.ndarray:
    """Embed text descriptions using CLIP."""
    embeddings = []
    for text in texts:
        # Truncate to fit CLIP's 77 token limit
        inputs = processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=77)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            embeddings.append(text_features.cpu().numpy())
    return np.vstack(embeddings).astype('float32')

def embed_image(image_path: str, model, processor, device) -> np.ndarray:
    """Embed an image using CLIP."""
    try:
        image = Image.open(image_path).convert('RGB')
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        return image_features.cpu().numpy().astype('float32')
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# ========================== CLASSIFIER ==========================
class RangoliClassifier:
    def __init__(self, config: RAGConfig = CONFIG):
        self.config = config
        self.df = None
        self.index = None
        self.model = None
        self.processor = None
        self.device = None
        self._initialized = False

    def initialize(self):
        if self._initialized:
            return

        self.model, self.processor, self.device = get_clip_model()
        self.df = load_category_dataset(self.config.csv_path)

        # Embed category descriptions
        print("\n🔄 Creating category embeddings...")
        prompts = self.df['clip_prompt'].tolist()
        embeddings = embed_texts(prompts, self.model, self.processor, self.device)

        # Build FAISS index
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

        self._initialized = True
        print("✅ Classifier ready!")

    def classify(self, image_path: str) -> Dict[str, Any]:
        """Classify an image into one of the 6 Rangoli categories."""
        if not self._initialized:
            self.initialize()

        # Embed image
        image_embedding = embed_image(image_path, self.model, self.processor, self.device)
        if image_embedding is None:
            return {"error": "Failed to process image"}

        # Search for best matching categories
        scores, indices = self.index.search(image_embedding, self.config.top_k)

        results = []
        for i, idx in enumerate(indices[0]):
            if idx != -1:
                row = self.df.iloc[idx]
                score = float(scores[0][i])
                results.append({
                    'category': row['Category'],
                    'state': row['State'],
                    'local_name': row['Local_Name'],
                    'score': round(score, 4),
                    'row': row.to_dict()
                })

        if self.config.debug:
            print(f"\n📊 CLASSIFICATION SCORES:")
            print("   " + "─" * 50)
            for r in results:
                status = "✅" if r['score'] >= self.config.confidence_threshold else "⚠️"
                print(f"   {status} {r['category']:15} ({r['state']:20}) → {r['score']:.4f}")
            print("   " + "─" * 50)

        return {
            'top_match': results[0] if results else None,
            'all_matches': results,
            'confidence': results[0]['score'] if results else 0
        }

    def generate_output(self, classification: Dict[str, Any]) -> Dict[str, Any]:
        """Generate structured JSON output from classification."""
        if 'error' in classification:
            return self._error_output(classification['error'])

        top = classification.get('top_match')
        if not top or top['score'] < self.config.confidence_threshold:
            return self._low_confidence_output(classification)

        row = top['row']
        score = top['score']

        # Map score to confidence
        if score >= 0.30:
            confidence = 0.95
        elif score >= 0.26:
            confidence = 0.90
        elif score >= 0.24:
            confidence = 0.85
        elif score >= 0.22:
            confidence = 0.80
        else:
            confidence = 0.60

        return {
            "Detected_Category": top['category'],
            "Detected_State": row.get('State', ''),
            "Local_Name": row.get('Local_Name', ''),
            "Confidence": round(confidence, 2),
            "Visual_Description": row.get('Visual_Description', ''),
            "Unique_Identifiers": row.get('Unique_Identifiers', ''),
            "Story": row.get('Story', ''),
            "Origin": row.get('Origin', ''),
            "Cultural_Significance": row.get('Cultural_Significance', ''),
            "Design_Pattern": row.get('Design_Pattern', ''),
            "Symmetry": row.get('Symmetry', ''),
            "Traditional_Colours": row.get('Traditional_Colours', ''),
            "Materials_Used": row.get('Materials_Used', ''),
            "Festive_Significance": row.get('Festive_Significance', ''),
            "Alternative_Matches": [
                {"category": m['category'], "state": m['state'], "score": m['score']}
                for m in classification['all_matches'][1:]
            ],
            "Raw_Score": top['score'],
            "Notes": f"Classified as {top['category']} with {confidence:.0%} confidence"
        }

    def _error_output(self, error: str) -> Dict[str, Any]:
        return {
            "Detected_Category": "Unknown",
            "Confidence": 0.0,
            "Notes": f"Error: {error}"
        }

    def _low_confidence_output(self, classification: Dict[str, Any]) -> Dict[str, Any]:
        matches = classification.get('all_matches', [])
        return {
            "Detected_Category": "Uncertain",
            "Confidence": matches[0]['score'] if matches else 0,
            "Possible_Categories": [
                {"category": m['category'], "score": m['score']}
                for m in matches
            ],
            "Notes": "Low confidence - image may not be a traditional Rangoli/Kolam, or is a hybrid style"
        }

    def process(self, image_path: str) -> str:
        """Full pipeline: classify and generate output."""
        classification = self.classify(image_path)
        output = self.generate_output(classification)
        return json.dumps(output, indent=2, ensure_ascii=False)

# ========================== ENHANCED WITH LLM ==========================
class EnhancedRangoliClassifier(RangoliClassifier):
    def generate_with_llm(self, classification: Dict[str, Any]) -> Dict[str, Any]:
        """Use LLM to enhance the classification output."""
        if 'error' in classification or not classification.get('top_match'):
            return self.generate_output(classification)

        top = classification['top_match']
        row = top['row']

        system_prompt = """You are an expert on Indian traditional floor art.
Given classification results, produce a detailed JSON response.
Use the matched category data but enrich with your knowledge.
Be accurate - don't invent facts not supported by the data.
OUTPUT valid JSON only."""

        user_prompt = f"""Classification result:
- Category: {top['category']} ({row.get('State', '')})
- Score: {top['score']}
- Visual Description: {row.get('Visual_Description', '')}
- Unique Identifiers: {row.get('Unique_Identifiers', '')}

Alternative matches: {json.dumps(classification['all_matches'][1:3], indent=2)}

Generate a comprehensive JSON with: Detected_Category, Detected_State, Local_Name, Confidence,
Visual_Description, Story, Cultural_Significance, Design_Pattern, Traditional_Colours,
Materials_Used, Festive_Significance, Alternative_Matches, Notes"""

        result = None
        if self.config.llm_provider == 'gemini' and self.config.gemini_api_key:
            result = self._call_gemini(system_prompt, user_prompt)

        if result:
            return result
        return self.generate_output(classification)

    def _call_gemini(self, system: str, user: str) -> Dict[str, Any]:
        """Call Google Gemini API"""
        try:
            import google.generativeai as genai
            genai.configure(api_key=self.config.gemini_api_key)

            model = genai.GenerativeModel(
                model_name=self.config.gemini_model,
                system_instruction=system
            )

            response = model.generate_content(
                user,
                generation_config=genai.types.GenerationConfig(
                    temperature=self.config.llm_temperature,
                    max_output_tokens=self.config.llm_max_tokens
                )
            )

            text = response.text.strip()
            # Remove markdown code blocks if present
            if text.startswith("```json"):
                text = text[7:]
            if text.startswith("```"):
                text = text[3:]
            if text.endswith("```"):
                text = text[:-3]
            text = text.strip()

            return json.loads(text)
        except Exception as e:
            print(f"⚠️ Gemini error: {e}")
            return None

    def process(self, image_path: str) -> str:
        classification = self.classify(image_path)
        output = self.generate_with_llm(classification)
        return json.dumps(output, indent=2, ensure_ascii=False)

# ========================== GLOBAL API ==========================
_CLASSIFIER = None

def initialize_classifier(
    csv_path: str = '/content/RangoliDataset_6cat.csv',
    llm_provider: str = 'none',
    gemini_api_key: str = ""
):
    """
    Initialize the Rangoli classifier.

    Args:
        csv_path: Path to the 6-category dataset CSV
        llm_provider: 'none' or 'gemini'
        gemini_api_key: Your Google Gemini API key (from aistudio.google.com)
    """
    global _CLASSIFIER
    CONFIG.csv_path = csv_path
    CONFIG.llm_provider = llm_provider
    CONFIG.gemini_api_key = gemini_api_key

    if llm_provider != 'none':
        _CLASSIFIER = EnhancedRangoliClassifier(CONFIG)
    else:
        _CLASSIFIER = RangoliClassifier(CONFIG)

    _CLASSIFIER.initialize()
    print(f"\n🎉 Classifier ready! (LLM: {llm_provider})")
    return _CLASSIFIER

def classify_rangoli(image_path: str) -> dict:
    """Classify a Rangoli/Kolam image."""
    global _CLASSIFIER
    if _CLASSIFIER is None:
        initialize_classifier()
    return json.loads(_CLASSIFIER.process(image_path))

def upload_and_classify():
    """Upload an image and classify it."""
    from google.colab import files
    print("\n📤 Upload a Rangoli/Kolam image (any design)...")
    uploaded = files.upload()
    if not uploaded:
        print("❌ No file uploaded.")
        return None
    filename = list(uploaded.keys())[0]
    print(f"\n🔍 Classifying: {filename}")
    result = classify_rangoli(filename)
    print("\n" + "=" * 60)
    print("📋 CLASSIFICATION RESULT")
    print("=" * 60)
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result

print("✅ CELL 1 COMPLETE!")
print("\n📚 This classifier recognizes 6 types of Indian floor art:")
print("   1. Mandana (Rajasthan) - White on red/ochre")
print("   2. Alpana (West Bengal) - White rice paste, freehand")
print("   3. Sathiya/Gahuli (Gujarat) - Swastika, mazes, rice grains")
print("   4. Rangoli (Maharashtra/Karnataka) - Colorful, gradients")
print("   5. Muggu (Andhra Pradesh) - Star patterns, colorful")
print("   6. Kolam (Tamil Nadu) - Dot-grid with loops")
print("\n📝 Run CELL 2 to start classifying images!")

✅ CELL 1 COMPLETE!

📚 This classifier recognizes 6 types of Indian floor art:
   1. Mandana (Rajasthan) - White on red/ochre
   2. Alpana (West Bengal) - White rice paste, freehand
   3. Sathiya/Gahuli (Gujarat) - Swastika, mazes, rice grains
   4. Rangoli (Maharashtra/Karnataka) - Colorful, gradients
   5. Muggu (Andhra Pradesh) - Star patterns, colorful
   6. Kolam (Tamil Nadu) - Dot-grid with loops

📝 Run CELL 2 to start classifying images!


# Run

In [ ]:
initialize_classifier(
    csv_path='/content/RangoliDataset_6cat.csv',
    llm_provider='none',
)
upload_and_classify()

🔄 Loading CLIP model...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✅ CLIP model loaded on cpu
📂 Loading category dataset from /content/RangoliDataset_6cat.csv...
✅ Loaded 6 categories:
   • Mandana (RAJASTHAN) - Mandana
   • Alpana (WEST BENGAL) - Alpana/Pithuli
   • Sathiya_Gahuli (GUJARAT) - Sathiya/Gahuli
   • Rangoli_MH_KA (MAHARASHTRA/KARNATAKA) - Rangoli/Rangavalli
   • Muggu (ANDHRA PRADESH/TELANGANA) - Muggu
   • Kolam (TAMIL NADU) - Kolam

🔄 Creating category embeddings...
✅ Classifier ready!

🎉 Classifier ready! (LLM: none)

📤 Upload a Rangoli/Kolam image (any design)...
